# Farm Yield Prediction - Model Training

This notebook trains the **CatBoost** classification model for crop prediction. It is optimized for Google Colab to utilize GPU acceleration if available.

In [ ]:
# 1. Install dependencies
!pip install catboost pandas numpy scikit-learn

In [ ]:
# 2. Import libraries
import pandas as pd
import os
import numpy as np
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import accuracy_score, classification_report
from google.colab import files

### 3. Upload your dataset
Upload `nigeria_agri_yield_enhanced.csv` from your local `data/` folder.

In [ ]:
uploaded = files.upload()
DATA_PATH = list(uploaded.keys())[0]
print(f"Using dataset: {DATA_PATH}")

In [ ]:
def engineer_features(df):
    """Advanced feature engineering including interactions."""
    print("Engineering high-dimensional features...")
    # 1. Base indices
    df['soil_fertility_index'] = df['soil_nitrogen'] + df['soil_phosphorus'] + df['soil_potassium']
    df['np_ratio'] = df['soil_nitrogen'] / (df['soil_phosphorus'] + 1e-6)
    df['climate_index'] = df['rainfall_mm'] * df['temperature_C']
    df['ph_stress'] = (df['soil_pH'] - 7.0).abs()
    
    # 2. Interactions
    df['n_ph_inter'] = df['soil_nitrogen'] * df['soil_pH']
    df['p_ph_inter'] = df['soil_phosphorus'] * df['soil_pH']
    df['rain_temp_ratio'] = df['rainfall_mm'] / (df['temperature_C'] + 1e-6)
    
    return df

In [ ]:
def train_model(data_path):
    df = pd.read_csv(data_path)
    df = engineer_features(df)

    categorical_features = [
        'region', 'state', 'agro_zone', 'soil_type', 
        'pest_type', 'pest_severity', 'rainfall_variability', 'labor_input'
    ]
    
    numeric_features = [
        'soil_nitrogen', 'soil_phosphorus', 'soil_potassium', 
        'soil_pH', 'temperature_C', 'rainfall_mm', 'humidity', 'farm_size_ha',
        'soil_fertility_index', 'np_ratio', 'climate_index', 'ph_stress',
        'n_ph_inter', 'p_ph_inter', 'rain_temp_ratio'
    ]
    
    target_col = 'crop_type'
    df_clean = df[categorical_features + numeric_features + [target_col]].dropna()

    X = df_clean[categorical_features + numeric_features]
    y = df_clean[target_col]

    # SPLIT
    X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Class weights
    classes = np.unique(y_train_full)
    weights = len(y_train_full) / (len(classes) * pd.Series(y_train_full).value_counts().sort_index().values)
    class_weights = dict(zip(classes, weights))

    # Train model (Using GPU if available)
    model = CatBoostClassifier(
        iterations=1000,
        depth=8,
        learning_rate=0.05,
        loss_function='MultiClass',
        cat_features=categorical_features,
        class_weights=class_weights,
        task_type='GPU' if os.environ.get('COLAB_GPU', '0') == '1' else 'CPU',
        random_seed=42,
        verbose=200
    )
    
    model.fit(X_train_full, y_train_full)

    # Evaluate
    y_pred = model.predict(X_test)
    print("\nFinal Classification Report:")
    print(classification_report(y_test, y_pred))

    # Save and Download
    model.save_model('crop_model.cbm')
    files.download('crop_model.cbm')

train_model(DATA_PATH)

### 4. Deployment
Once the training is finished, Colab will automatically prompt you to download `crop_model.cbm`. Move this file back into your local project's `models/` directory.